# DEEPLENSE9 — Unsupervised Super-Resolution of Gravitational Lensing Images
## GSoC 2026 — ML4Sci | DeepLense Project

| Detail | Info |
|---|---|
| **Project** | Unsupervised Super-Resolution and Analysis of Real Lensing Images |
| **Mentors** | Michael Toomey (MIT), Pranath Reddy, Sergei Gleyzer (Alabama) |
| **Tests** | Test VI.A (Simulated SR) + Test VI.B (Real HSC/HST SR) |
| **Metrics** | MSE, SSIM, PSNR |

## Strategy
- **Task VI.A**: Train an SRCNN + ESRGAN-style model on simulated HR/LR lensing image pairs
- **Task VI.B**: Fine-tune the Task VI.A model on 300 real HSC/HST telescope image pairs using transfer learning
- Physics connection: Super-resolution directly enhances lensing substructure visibility, enabling better dark matter studies

In [ ]:
# CELL 1 — Install Libraries
!pip install torch torchvision matplotlib scikit-image scikit-learn numpy tqdm gdown Pillow seaborn

In [ ]:
# CELL 2 — Imports & Setup
import os
import glob
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from skimage.metrics import structural_similarity as ssim_metric
from skimage.metrics import peak_signal_noise_ratio as psnr_metric
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
# CELL 3 — Download Datasets
import gdown

os.makedirs('data/simulated', exist_ok=True)
os.makedirs('data/real', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

# Task VI.A — Simulated HR/LR lensing images
SIM_FILE = 'data/simulated/SR_dataset.zip'
if not os.path.exists(SIM_FILE):
    print('Downloading simulated SR dataset...')
    gdown.download(
        'https://drive.google.com/uc?id=1uJmDZw649XS-r-dYs9WD-OPwF_TIroVw',
        SIM_FILE, quiet=False
    )
    import zipfile
    with zipfile.ZipFile(SIM_FILE, 'r') as z:
        z.extractall('data/simulated/')
    print('Simulated dataset ready!')

# Task VI.B — Real HSC/HST telescope images (300 pairs)
REAL_FILE = 'data/real/real_SR_dataset.zip'
if not os.path.exists(REAL_FILE):
    print('Downloading real HSC/HST dataset...')
    gdown.download(
        'https://drive.google.com/uc?id=1plYfM-jFJT7TbTMVssuCCFvLzGdxMQ4h',
        REAL_FILE, quiet=False
    )
    import zipfile
    with zipfile.ZipFile(REAL_FILE, 'r') as z:
        z.extractall('data/real/')
    print('Real dataset ready!')

# Explore folder structure
for root, dirs, files in os.walk('data/'):
    level = root.replace('data/', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f'{indent}{os.path.basename(root)}/ ({len(files)} files)')
    if level > 2:
        break

In [ ]:
# CELL 4 — Dataset Classes

def load_image(path):
    """Load image from .npy or image file, return numpy array."""
    if path.endswith('.npy'):
        img = np.load(path, allow_pickle=True)
        if img.dtype == object:
            img = np.stack([np.array(r) for r in img]).astype(np.float32)
        else:
            img = img.astype(np.float32)
    else:
        img = np.array(Image.open(path).convert('L'), dtype=np.float32)
    img = np.squeeze(img)
    # Normalize to [0, 1]
    img = (img - img.min()) / (img.max() - img.min() + 1e-8)
    return img


class SRDataset(Dataset):
    """
    Super-Resolution Dataset.
    Loads HR/LR image pairs.
    If only HR images are available, generates LR by downsampling.
    """
    def __init__(self, hr_paths, lr_paths=None, hr_size=64, scale=4, augment=False):
        self.hr_paths  = hr_paths
        self.lr_paths  = lr_paths
        self.hr_size   = hr_size
        self.lr_size   = hr_size // scale
        self.scale     = scale
        self.augment   = augment
        print(f'SRDataset: {len(hr_paths)} pairs | HR={hr_size}x{hr_size} | LR={self.lr_size}x{self.lr_size}')

    def __len__(self):
        return len(self.hr_paths)

    def __getitem__(self, idx):
        # Load HR image
        hr = load_image(self.hr_paths[idx])
        hr = torch.from_numpy(hr).unsqueeze(0).float()  # (1, H, W)
        hr = TF.resize(hr, [self.hr_size, self.hr_size], antialias=True)

        # Load or generate LR image
        if self.lr_paths is not None:
            lr = load_image(self.lr_paths[idx])
            lr = torch.from_numpy(lr).unsqueeze(0).float()
        else:
            # Downsample HR to get LR (bicubic degradation)
            lr = TF.resize(hr, [self.lr_size, self.lr_size], antialias=True)

        lr = TF.resize(lr, [self.lr_size, self.lr_size], antialias=True)

        # Augmentation
        if self.augment:
            if random.random() > 0.5:
                hr = TF.hflip(hr)
                lr = TF.hflip(lr)
            if random.random() > 0.5:
                hr = TF.vflip(hr)
                lr = TF.vflip(lr)
            k = random.randint(0, 3)
            hr = torch.rot90(hr, k, dims=[1, 2])
            lr = torch.rot90(lr, k, dims=[1, 2])

        return lr, hr


def find_image_pairs(data_dir):
    """Find HR and LR image pairs in a directory."""
    hr_paths, lr_paths = [], []

    # Look for HR/LR subdirectories
    hr_dir = None
    lr_dir = None
    for dirpath, dirs, files in os.walk(data_dir):
        dname = os.path.basename(dirpath).lower()
        if 'hr' in dname or 'high' in dname:
            hr_dir = dirpath
        if 'lr' in dname or 'low' in dname:
            lr_dir = dirpath

    if hr_dir and lr_dir:
        hr_files = sorted(glob.glob(f'{hr_dir}/*.npy') + glob.glob(f'{hr_dir}/*.png'))
        lr_files = sorted(glob.glob(f'{lr_dir}/*.npy') + glob.glob(f'{lr_dir}/*.png'))
        n = min(len(hr_files), len(lr_files))
        hr_paths = hr_files[:n]
        lr_paths = lr_files[:n]
        print(f'Found {n} HR/LR pairs in {hr_dir} and {lr_dir}')
    else:
        # Fallback: use all files as HR, generate LR on the fly
        hr_paths = sorted(glob.glob(f'{data_dir}/**/*.npy', recursive=True) +
                          glob.glob(f'{data_dir}/**/*.png', recursive=True))
        lr_paths = None
        print(f'Found {len(hr_paths)} HR images (LR will be generated by downsampling)')

    return hr_paths, lr_paths

print('Dataset class ready!')

In [ ]:
# CELL 5 — Load & Visualize Datasets

# Task VI.A — Simulated
hr_sim, lr_sim = find_image_pairs('data/simulated/')
print(f'Simulated: {len(hr_sim)} pairs')

# Task VI.B — Real
hr_real, lr_real = find_image_pairs('data/real/')
print(f'Real: {len(hr_real)} pairs')

# Build train/val splits
hr_tr, hr_val, lr_tr, lr_val = train_test_split(
    hr_sim, lr_sim if lr_sim else hr_sim,
    test_size=0.1, random_state=SEED
)

HR_SIZE = 64
SCALE   = 4   # 4x super-resolution

train_dataset = SRDataset(hr_tr,  lr_tr  if lr_sim else None, hr_size=HR_SIZE, scale=SCALE, augment=True)
val_dataset   = SRDataset(hr_val, lr_val if lr_sim else None, hr_size=HR_SIZE, scale=SCALE, augment=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

# Visualize samples
lr_sample, hr_sample = next(iter(val_loader))
fig, axes = plt.subplots(2, 8, figsize=(20, 5))
for i in range(8):
    axes[0, i].imshow(lr_sample[i, 0].numpy(), cmap='inferno')
    axes[0, i].set_title('LR Input', fontsize=8)
    axes[0, i].axis('off')
    axes[1, i].imshow(hr_sample[i, 0].numpy(), cmap='inferno')
    axes[1, i].set_title('HR Target', fontsize=8)
    axes[1, i].axis('off')
plt.suptitle('Task VI.A — Simulated Lensing Images: LR vs HR', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/01_dataset_samples.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/01_dataset_samples.png')

In [ ]:
# CELL 6 — SR Model: SRCNN + Residual Upsampler
# Architecture: Residual Channel Attention Network (RCAN-lite)
# Why: Best balance of performance and efficiency for scientific images

class ResidualBlock(nn.Module):
    """Residual block with channel attention for fine-grained feature learning."""
    def __init__(self, channels=64, reduction=16):
        super().__init__()
        self.conv1  = nn.Conv2d(channels, channels, 3, padding=1)
        self.relu   = nn.ReLU(inplace=True)
        self.conv2  = nn.Conv2d(channels, channels, 3, padding=1)
        # Channel attention
        self.ca = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Linear(channels, channels // reduction),
            nn.ReLU(),
            nn.Linear(channels // reduction, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        res = self.conv2(self.relu(self.conv1(x)))
        ca  = self.ca(res).view(res.shape[0], res.shape[1], 1, 1)
        return x + res * ca


class LensingSRNet(nn.Module):
    """
    Super-Resolution Network for Gravitational Lensing Images.

    Architecture:
    1. Shallow feature extraction
    2. Deep residual blocks with channel attention
    3. Pixel shuffle upsampling (sub-pixel convolution)
    4. Reconstruction head

    Physics motivation: Channel attention learns to weight
    spectral features that correspond to lensing substructure
    signatures — directly analogous to feature selection in
    physics-informed models.
    """
    def __init__(self, scale=4, n_channels=64, n_blocks=8):
        super().__init__()
        self.scale = scale

        # Shallow feature extraction
        self.head = nn.Conv2d(1, n_channels, 3, padding=1)

        # Deep feature extraction with residual blocks
        self.body = nn.Sequential(*[ResidualBlock(n_channels) for _ in range(n_blocks)])
        self.body_end = nn.Conv2d(n_channels, n_channels, 3, padding=1)

        # Upsampling: pixel shuffle (sub-pixel convolution)
        # For scale=4: two 2x upsamplers
        upsamplers = []
        s = scale
        while s > 1:
            upsamplers += [
                nn.Conv2d(n_channels, n_channels * 4, 3, padding=1),
                nn.PixelShuffle(2),
                nn.ReLU(inplace=True),
            ]
            s //= 2
        self.upsample = nn.Sequential(*upsamplers)

        # Reconstruction
        self.tail = nn.Conv2d(n_channels, 1, 3, padding=1)

    def forward(self, x):
        feat      = self.head(x)
        body_out  = self.body_end(self.body(feat))
        feat      = feat + body_out        # global residual
        feat      = self.upsample(feat)
        return torch.sigmoid(self.tail(feat))  # output in [0,1]


# Bicubic baseline (for comparison)
class BicubicUpsampler(nn.Module):
    def __init__(self, scale=4):
        super().__init__()
        self.scale = scale

    def forward(self, x):
        h, w = x.shape[2] * self.scale, x.shape[3] * self.scale
        return torch.nn.functional.interpolate(x, size=(h, w), mode='bicubic', align_corners=False)


model   = LensingSRNet(scale=SCALE, n_channels=64, n_blocks=8).to(DEVICE)
bicubic = BicubicUpsampler(scale=SCALE).to(DEVICE)

total   = sum(p.numel() for p in model.parameters())
print(f'LensingSRNet params: {total:,}')

# Smoke test
lr_test = torch.randn(2, 1, HR_SIZE // SCALE, HR_SIZE // SCALE).to(DEVICE)
with torch.no_grad():
    sr_test = model(lr_test)
print(f'Input:  {lr_test.shape}')
print(f'Output: {sr_test.shape}')
print(f'Expected: (2, 1, {HR_SIZE}, {HR_SIZE})')

In [ ]:
# CELL 7 — Loss Functions

class CombinedSRLoss(nn.Module):
    """
    Combined loss for super-resolution:
    - L1 loss: pixel-level accuracy
    - Perceptual loss: structural similarity
    - Gradient loss: preserve edge sharpness (critical for lensing arcs)
    """
    def __init__(self, l1_w=1.0, grad_w=0.1):
        super().__init__()
        self.l1_w   = l1_w
        self.grad_w = grad_w
        self.l1     = nn.L1Loss()
        # Sobel filter for gradient loss
        sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32)
        sobel_y = torch.tensor([[-1,-2,-1], [ 0, 0, 0], [ 1, 2, 1]], dtype=torch.float32)
        self.register_buffer = lambda n, t: setattr(self, n, t)
        self.sobel_x = sobel_x.view(1, 1, 3, 3)
        self.sobel_y = sobel_y.view(1, 1, 3, 3)

    def gradient(self, x):
        sx = self.sobel_x.to(x.device)
        sy = self.sobel_y.to(x.device)
        gx = torch.nn.functional.conv2d(x, sx, padding=1)
        gy = torch.nn.functional.conv2d(x, sy, padding=1)
        return torch.sqrt(gx**2 + gy**2 + 1e-8)

    def forward(self, sr, hr):
        l1_loss   = self.l1(sr, hr)
        grad_loss = self.l1(self.gradient(sr), self.gradient(hr))
        return self.l1_w * l1_loss + self.grad_w * grad_loss


criterion = CombinedSRLoss(l1_w=1.0, grad_w=0.1)
print('Loss functions ready!')

In [ ]:
# CELL 8 — Evaluation Metrics

def compute_metrics(sr_batch, hr_batch):
    """Compute MSE, SSIM, PSNR on a batch of images."""
    mse_list, ssim_list, psnr_list = [], [], []

    sr_np = sr_batch.cpu().numpy()
    hr_np = hr_batch.cpu().numpy()

    for i in range(len(sr_np)):
        sr_img = sr_np[i, 0].clip(0, 1)
        hr_img = hr_np[i, 0].clip(0, 1)

        mse  = np.mean((sr_img - hr_img) ** 2)
        ssim = ssim_metric(sr_img, hr_img, data_range=1.0)
        psnr = psnr_metric(hr_img, sr_img, data_range=1.0)

        mse_list.append(mse)
        ssim_list.append(ssim)
        psnr_list.append(psnr)

    return np.mean(mse_list), np.mean(ssim_list), np.mean(psnr_list)


@torch.no_grad()
def evaluate(model, loader, desc='Eval'):
    model.eval()
    all_mse, all_ssim, all_psnr = [], [], []

    for lr_imgs, hr_imgs in tqdm(loader, desc=desc, leave=False):
        lr_imgs = lr_imgs.to(DEVICE)
        hr_imgs = hr_imgs.to(DEVICE)
        sr_imgs = model(lr_imgs)
        # Resize SR to match HR if needed
        if sr_imgs.shape != hr_imgs.shape:
            sr_imgs = torch.nn.functional.interpolate(
                sr_imgs, size=hr_imgs.shape[2:], mode='bilinear', align_corners=False)
        mse, ssim, psnr = compute_metrics(sr_imgs, hr_imgs)
        all_mse.append(mse)
        all_ssim.append(ssim)
        all_psnr.append(psnr)

    return np.mean(all_mse), np.mean(all_ssim), np.mean(all_psnr)

print('Metrics functions ready!')

In [ ]:
# CELL 9 — Task VI.A Training: Simulated SR

EPOCHS_A   = 50
LR_A       = 2e-4

optimizer  = optim.Adam(model.parameters(), lr=LR_A, betas=(0.9, 0.999))
scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS_A, eta_min=1e-6)

train_losses = []
val_metrics  = {'mse': [], 'ssim': [], 'psnr': []}
best_psnr    = 0

print(f'Training Task VI.A for {EPOCHS_A} epochs...')
for epoch in range(EPOCHS_A):
    model.train()
    ep_loss = 0

    for lr_imgs, hr_imgs in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS_A}', leave=False):
        lr_imgs = lr_imgs.to(DEVICE)
        hr_imgs = hr_imgs.to(DEVICE)

        sr_imgs = model(lr_imgs)
        if sr_imgs.shape != hr_imgs.shape:
            sr_imgs = torch.nn.functional.interpolate(
                sr_imgs, size=hr_imgs.shape[2:], mode='bilinear', align_corners=False)

        loss = criterion(sr_imgs, hr_imgs)
        optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        ep_loss += loss.item()

    scheduler.step()
    avg_loss = ep_loss / len(train_loader)
    train_losses.append(avg_loss)

    # Evaluate every 5 epochs
    if (epoch + 1) % 5 == 0:
        mse, ssim, psnr = evaluate(model, val_loader, desc='Val')
        val_metrics['mse'].append(mse)
        val_metrics['ssim'].append(ssim)
        val_metrics['psnr'].append(psnr)
        print(f'Epoch {epoch+1:3d}/{EPOCHS_A} | Loss: {avg_loss:.4f} | MSE: {mse:.4f} | SSIM: {ssim:.4f} | PSNR: {psnr:.2f}dB')

        if psnr > best_psnr:
            best_psnr = psnr
            torch.save(model.state_dict(), 'outputs/best_sr_model_simulated.pth')
            print(f'  New best PSNR: {best_psnr:.2f}dB — model saved!')

print(f'Training complete! Best PSNR: {best_psnr:.2f}dB')

In [ ]:
# CELL 10 — Task VI.A Results Visualization

# Load best model
model.load_state_dict(torch.load('outputs/best_sr_model_simulated.pth', map_location=DEVICE, weights_only=True))
model.eval()

# Final metrics
mse_final, ssim_final, psnr_final = evaluate(model, val_loader, desc='Final Eval')

# Bicubic baseline
mse_bic, ssim_bic, psnr_bic = evaluate(bicubic, val_loader, desc='Bicubic')

print('='*55)
print('TASK VI.A RESULTS — Simulated Lensing SR')
print('='*55)
print(f'Method        MSE       SSIM      PSNR')
print(f'Bicubic     {mse_bic:.4f}    {ssim_bic:.4f}    {psnr_bic:.2f}dB')
print(f'LensingSRNet {mse_final:.4f}    {ssim_final:.4f}    {psnr_final:.2f}dB')
print('='*55)

# Visual comparison: LR | Bicubic | SR Model | HR
lr_batch, hr_batch = next(iter(val_loader))
lr_batch = lr_batch.to(DEVICE)
with torch.no_grad():
    sr_batch  = model(lr_batch)
    bic_batch = bicubic(lr_batch)
    if sr_batch.shape != hr_batch.shape:
        sr_batch  = torch.nn.functional.interpolate(sr_batch,  size=hr_batch.shape[2:], mode='bilinear', align_corners=False)
        bic_batch = torch.nn.functional.interpolate(bic_batch, size=hr_batch.shape[2:], mode='bilinear', align_corners=False)

N = 4
fig, axes = plt.subplots(N, 4, figsize=(16, N*4))
titles = ['LR Input', 'Bicubic', 'LensingSRNet (Ours)', 'HR Ground Truth']
batches = [lr_batch, bic_batch, sr_batch, hr_batch.to(DEVICE)]

for row in range(N):
    for col, (title, batch) in enumerate(zip(titles, batches)):
        img = batch[row, 0].cpu().numpy().clip(0, 1)
        axes[row, col].imshow(img, cmap='inferno')
        if row == 0:
            axes[row, col].set_title(title, fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

plt.suptitle(f'Task VI.A — Simulated SR | PSNR: Bicubic={psnr_bic:.2f}dB vs Ours={psnr_final:.2f}dB',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/02_task_a_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/02_task_a_results.png')

In [ ]:
# CELL 11 — Training Loss & Metrics Plots

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_eval = list(range(5, EPOCHS_A + 1, 5))

axes[0].plot(train_losses, color='steelblue', linewidth=2)
axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('L1 + Gradient Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_eval, val_metrics['ssim'], color='green', linewidth=2, marker='o')
axes[1].axhline(y=ssim_bic, color='red', linestyle='--', label=f'Bicubic: {ssim_bic:.4f}')
axes[1].set_title('Validation SSIM', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('SSIM')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_eval, val_metrics['psnr'], color='orange', linewidth=2, marker='o')
axes[2].axhline(y=psnr_bic, color='red', linestyle='--', label=f'Bicubic: {psnr_bic:.2f}dB')
axes[2].set_title('Validation PSNR', fontsize=13, fontweight='bold')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('PSNR (dB)')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Task VI.A Training Curves', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/03_training_curves.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/03_training_curves.png')

---
# Task VI.B — Real HSC/HST Telescope Images
### Strategy: Transfer Learning + Fine-tuning
- Load pre-trained model from Task VI.A
- Fine-tune on 300 real telescope image pairs
- Use lower learning rate to preserve learned features
- Domain adaptation: real images have different noise characteristics than simulated

In [ ]:
# CELL 12 — Task VI.B: Load Real Dataset & Fine-tune

# Load real HSC/HST pairs
hr_real_paths, lr_real_paths = find_image_pairs('data/real/')
print(f'Real pairs found: {len(hr_real_paths)}')

# Train/val split (90:10)
hr_real_tr, hr_real_val, lr_real_tr_split, lr_real_val_split = train_test_split(
    hr_real_paths,
    lr_real_paths if lr_real_paths else hr_real_paths,
    test_size=0.1, random_state=SEED
)

real_train = SRDataset(hr_real_tr,  lr_real_tr_split  if lr_real_paths else None, hr_size=HR_SIZE, scale=SCALE, augment=True)
real_val   = SRDataset(hr_real_val, lr_real_val_split if lr_real_paths else None, hr_size=HR_SIZE, scale=SCALE, augment=False)

real_train_loader = DataLoader(real_train, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
real_val_loader   = DataLoader(real_val,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

print(f'Real Train: {len(real_train)} | Real Val: {len(real_val)}')

# Load pre-trained model from Task VI.A
model_finetune = LensingSRNet(scale=SCALE, n_channels=64, n_blocks=8).to(DEVICE)
model_finetune.load_state_dict(torch.load('outputs/best_sr_model_simulated.pth',
                                           map_location=DEVICE, weights_only=True))
print('Pre-trained model loaded from Task VI.A!')

# Fine-tuning with lower LR
EPOCHS_B  = 30
LR_B      = 5e-5   # 4x lower than Task A

opt_ft    = optim.Adam(model_finetune.parameters(), lr=LR_B, betas=(0.9, 0.999))
sch_ft    = optim.lr_scheduler.CosineAnnealingLR(opt_ft, T_max=EPOCHS_B, eta_min=1e-7)

ft_losses = []
ft_metrics = {'mse': [], 'ssim': [], 'psnr': []}
best_psnr_real = 0

print(f'Fine-tuning Task VI.B for {EPOCHS_B} epochs...')
for epoch in range(EPOCHS_B):
    model_finetune.train()
    ep_loss = 0

    for lr_imgs, hr_imgs in tqdm(real_train_loader, desc=f'FT Epoch {epoch+1}/{EPOCHS_B}', leave=False):
        lr_imgs = lr_imgs.to(DEVICE)
        hr_imgs = hr_imgs.to(DEVICE)

        sr_imgs = model_finetune(lr_imgs)
        if sr_imgs.shape != hr_imgs.shape:
            sr_imgs = torch.nn.functional.interpolate(
                sr_imgs, size=hr_imgs.shape[2:], mode='bilinear', align_corners=False)

        loss = criterion(sr_imgs, hr_imgs)
        opt_ft.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(model_finetune.parameters(), 1.0)
        opt_ft.step()
        ep_loss += loss.item()

    sch_ft.step()
    avg_loss = ep_loss / max(1, len(real_train_loader))
    ft_losses.append(avg_loss)

    if (epoch + 1) % 5 == 0:
        mse, ssim, psnr = evaluate(model_finetune, real_val_loader, desc='Real Val')
        ft_metrics['mse'].append(mse)
        ft_metrics['ssim'].append(ssim)
        ft_metrics['psnr'].append(psnr)
        print(f'FT Epoch {epoch+1:3d}/{EPOCHS_B} | Loss: {avg_loss:.4f} | MSE: {mse:.4f} | SSIM: {ssim:.4f} | PSNR: {psnr:.2f}dB')

        if psnr > best_psnr_real:
            best_psnr_real = psnr
            torch.save(model_finetune.state_dict(), 'outputs/best_sr_model_real.pth')
            print(f'  New best PSNR (real): {best_psnr_real:.2f}dB — model saved!')

print(f'Fine-tuning complete! Best PSNR on real data: {best_psnr_real:.2f}dB')

In [ ]:
# CELL 13 — Task VI.B Results Visualization

model_finetune.load_state_dict(torch.load('outputs/best_sr_model_real.pth',
                                           map_location=DEVICE, weights_only=True))
model_finetune.eval()

# Final metrics on real data
mse_real, ssim_real, psnr_real = evaluate(model_finetune, real_val_loader, desc='Real Final')
mse_bic_r, ssim_bic_r, psnr_bic_r = evaluate(bicubic, real_val_loader, desc='Bicubic Real')

print('='*55)
print('TASK VI.B RESULTS — Real HSC/HST Lensing SR')
print('='*55)
print(f'Method         MSE       SSIM      PSNR')
print(f'Bicubic      {mse_bic_r:.4f}    {ssim_bic_r:.4f}    {psnr_bic_r:.2f}dB')
print(f'Fine-tuned   {mse_real:.4f}    {ssim_real:.4f}    {psnr_real:.2f}dB')
print('='*55)

# Visual comparison on real images
lr_batch_r, hr_batch_r = next(iter(real_val_loader))
lr_batch_r = lr_batch_r.to(DEVICE)
with torch.no_grad():
    sr_batch_r  = model_finetune(lr_batch_r)
    bic_batch_r = bicubic(lr_batch_r)
    if sr_batch_r.shape != hr_batch_r.shape:
        sr_batch_r  = torch.nn.functional.interpolate(sr_batch_r,  size=hr_batch_r.shape[2:], mode='bilinear', align_corners=False)
        bic_batch_r = torch.nn.functional.interpolate(bic_batch_r, size=hr_batch_r.shape[2:], mode='bilinear', align_corners=False)

N = 4
fig, axes = plt.subplots(N, 4, figsize=(16, N*4))
titles = ['LR Input', 'Bicubic', 'Fine-tuned Model', 'HR Ground Truth']
batches_r = [lr_batch_r, bic_batch_r, sr_batch_r, hr_batch_r.to(DEVICE)]

for row in range(N):
    for col, (title, batch) in enumerate(zip(titles, batches_r)):
        img = batch[row, 0].cpu().numpy().clip(0, 1)
        axes[row, col].imshow(img, cmap='inferno')
        if row == 0:
            axes[row, col].set_title(title, fontsize=12, fontweight='bold')
        axes[row, col].axis('off')

plt.suptitle(f'Task VI.B — Real HSC/HST SR | PSNR: Bicubic={psnr_bic_r:.2f}dB vs Ours={psnr_real:.2f}dB',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/04_task_b_results.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/04_task_b_results.png')

In [ ]:
# CELL 14 — Final Summary Dashboard

fig = plt.figure(figsize=(18, 10))
gs  = gridspec.GridSpec(2, 3, figure=fig)

# Task A metrics bar chart
ax1 = fig.add_subplot(gs[0, 0])
metrics_names = ['MSE', 'SSIM', 'PSNR (dB)']
bic_vals = [mse_bic,   ssim_bic,   psnr_bic]
our_vals = [mse_final, ssim_final, psnr_final]
x = np.arange(len(metrics_names))
ax1.bar(x - 0.2, bic_vals, 0.4, label='Bicubic', color='salmon')
ax1.bar(x + 0.2, our_vals, 0.4, label='LensingSRNet', color='steelblue')
ax1.set_xticks(x)
ax1.set_xticklabels(metrics_names)
ax1.set_title('Task VI.A — Simulated SR\nBicubic vs LensingSRNet', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Task B metrics bar chart
ax2 = fig.add_subplot(gs[0, 1])
bic_r_vals = [mse_bic_r, ssim_bic_r, psnr_bic_r]
ft_vals    = [mse_real,  ssim_real,  psnr_real]
ax2.bar(x - 0.2, bic_r_vals, 0.4, label='Bicubic', color='salmon')
ax2.bar(x + 0.2, ft_vals,    0.4, label='Fine-tuned', color='green')
ax2.set_xticks(x)
ax2.set_xticklabels(metrics_names)
ax2.set_title('Task VI.B — Real HSC/HST SR\nBicubic vs Fine-tuned', fontweight='bold')
ax2.legend()
ax2.grid(True, alpha=0.3)

# PSNR improvement
ax3 = fig.add_subplot(gs[0, 2])
methods = ['Bicubic\n(Simulated)', 'LensingSRNet\n(Simulated)', 'Bicubic\n(Real)', 'Fine-tuned\n(Real)']
psnr_vals = [psnr_bic, psnr_final, psnr_bic_r, psnr_real]
colors = ['salmon', 'steelblue', 'lightsalmon', 'green']
bars = ax3.bar(methods, psnr_vals, color=colors, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, psnr_vals):
    ax3.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.1,
             f'{val:.2f}dB', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax3.set_title('PSNR Comparison — All Methods', fontweight='bold')
ax3.set_ylabel('PSNR (dB) — Higher is Better')
ax3.grid(True, alpha=0.3, axis='y')

# Training curves
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(train_losses, color='steelblue', label='Task A (Simulated)')
ax4.plot(range(len(ft_losses)), ft_losses, color='green', label='Task B (Fine-tune)')
ax4.set_title('Training Loss Curves', fontweight='bold')
ax4.set_xlabel('Epoch')
ax4.set_ylabel('Loss')
ax4.legend()
ax4.grid(True, alpha=0.3)

# SSIM curves
ax5 = fig.add_subplot(gs[1, 1])
epochs_a = list(range(5, EPOCHS_A + 1, 5))
epochs_b = list(range(5, EPOCHS_B + 1, 5))
ax5.plot(epochs_a, val_metrics['ssim'], color='steelblue', marker='o', label='Task A Val SSIM')
if ft_metrics['ssim']:
    ax5.plot(epochs_b, ft_metrics['ssim'], color='green', marker='s', label='Task B Val SSIM')
ax5.set_title('Validation SSIM', fontweight='bold')
ax5.set_xlabel('Epoch')
ax5.set_ylabel('SSIM')
ax5.legend()
ax5.grid(True, alpha=0.3)

# Summary table
ax6 = fig.add_subplot(gs[1, 2])
ax6.axis('off')
table_data = [
    ['Task', 'Method', 'MSE', 'SSIM', 'PSNR'],
    ['VI.A', 'Bicubic',     f'{mse_bic:.4f}',   f'{ssim_bic:.4f}',   f'{psnr_bic:.2f}dB'],
    ['VI.A', 'LensingSRNet',f'{mse_final:.4f}',  f'{ssim_final:.4f}', f'{psnr_final:.2f}dB'],
    ['VI.B', 'Bicubic',     f'{mse_bic_r:.4f}',  f'{ssim_bic_r:.4f}',f'{psnr_bic_r:.2f}dB'],
    ['VI.B', 'Fine-tuned',  f'{mse_real:.4f}',   f'{ssim_real:.4f}', f'{psnr_real:.2f}dB'],
]
table = ax6.table(cellText=table_data[1:], colLabels=table_data[0],
                  cellLoc='center', loc='center',
                  colColours=['lightblue']*5)
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.8)
ax6.set_title('Complete Results Summary', fontweight='bold')

plt.suptitle('DEEPLENSE9 — Super-Resolution Results Dashboard', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('outputs/05_final_dashboard.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: outputs/05_final_dashboard.png')

In [ ]:
# CELL 15 — Final Summary

print('='*60)
print('  DEEPLENSE9 — COMPLETE RESULTS SUMMARY')
print('='*60)
print()
print('TASK VI.A — Simulated Lensing Super-Resolution')
print('-'*50)
print(f'  Bicubic Baseline : MSE={mse_bic:.4f} | SSIM={ssim_bic:.4f} | PSNR={psnr_bic:.2f}dB')
print(f'  LensingSRNet     : MSE={mse_final:.4f} | SSIM={ssim_final:.4f} | PSNR={psnr_final:.2f}dB')
print(f'  PSNR Improvement : +{psnr_final - psnr_bic:.2f}dB over bicubic')
print()
print('TASK VI.B — Real HSC/HST Telescope SR (Transfer Learning)')
print('-'*50)
print(f'  Bicubic Baseline : MSE={mse_bic_r:.4f} | SSIM={ssim_bic_r:.4f} | PSNR={psnr_bic_r:.2f}dB')
print(f'  Fine-tuned Model : MSE={mse_real:.4f} | SSIM={ssim_real:.4f} | PSNR={psnr_real:.2f}dB')
print(f'  PSNR Improvement : +{psnr_real - psnr_bic_r:.2f}dB over bicubic')
print()
print('ARCHITECTURE: Residual Channel Attention Network (RCAN-lite)')
print('  - 8 residual blocks with channel attention')
print('  - Pixel shuffle 4x upsampling')
print('  - Combined L1 + gradient loss')
print()
print('CONNECTION TO DEEPLENSE:')
print('  - Built on my DEEPLENSE4 Foundation Model work')
print('  - SR directly enhances lensing substructure visibility')
print('  - Real telescope adaptation enables LSST/Euclid pipeline use')
print('='*60)
print()
print('Output files:')
for f in sorted(os.listdir('outputs/')):
    print(f'  outputs/{f}')